# Embeddings and Chroma Vector Store

This notebook corresponds to `src/embeddings/build_chroma.py`.

Purpose:

```text
Data/chunks.json  →  Data/chroma_db/
```

It embeds each chunk with `sentence-transformers/all-MiniLM-L6-v2` and stores embeddings, text, and metadata in a persistent Chroma collection.

In [ ]:
from pathlib import Path
import json
import chromadb
from sentence_transformers import SentenceTransformer

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_PATH = PROJECT_ROOT / "Data" / "chunks.json"
CHROMA_DIR = PROJECT_ROOT / "Data" / "chroma_db"

print("Chunks path exists:", CHUNKS_PATH.exists())
print("Chroma directory:", CHROMA_DIR)

In [ ]:
def load_chunks(chunks_path: Path) -> list:
    with chunks_path.open("r", encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def build_chroma_vectorstore(
    chunks_path: Path,
    chroma_dir: Path,
    collection_name: str = "rag_documents",
    model_name: str = "all-MiniLM-L6-v2",
    reset_collection: bool = True,
):

    chunks = load_chunks(chunks_path)

    print(f"Loaded {len(chunks)} chunks.")
    print(f"Loading embedding model: {model_name}")

    model = SentenceTransformer(model_name)

    client = chromadb.PersistentClient(path=str(chroma_dir))

    if reset_collection:
        try:
            client.delete_collection(name=collection_name)
            print(f"Deleted existing collection: {collection_name}")
        except Exception:
            pass

    collection = client.get_or_create_collection(name=collection_name)

    ids = []
    texts = []
    metadatas = []

    for chunk in chunks:
        ids.append(chunk["chunk_id"])
        texts.append(chunk["text"])
        metadatas.append({
            "doc_id": chunk["doc_id"],
            "title": chunk["title"],
            "source_html": chunk["source_html"],
        })

    print("Creating embeddings...")
    embeddings = model.encode(
        texts,
        show_progress_bar=True,
        convert_to_numpy=True,
    ).tolist()

    print("Adding embeddings to Chroma...")

    batch_size = 500

    for start in range(0, len(ids), batch_size):
        end = start + batch_size

        collection.add(
            ids=ids[start:end],
            documents=texts[start:end],
            embeddings=embeddings[start:end],
            metadatas=metadatas[start:end],
        )

        print(f"Inserted {min(end, len(ids))}/{len(ids)} chunks.")

    print(f"Chroma vector store saved to {chroma_dir}")
    print(f"Collection name: {collection_name}")

    return collection

In [ ]:
collection = build_chroma_vectorstore(
    chunks_path=CHUNKS_PATH,
    chroma_dir=CHROMA_DIR,
    collection_name="rag_documents",
    model_name="all-MiniLM-L6-v2",
    reset_collection=True,
)

In [ ]:
print("Collection count:", collection.count())